# 04 — Mitigation: GRU + Reweighting (Pre-processing)

Applies **importance-weight reweighting** to the GRU baseline to reduce label-frequency bias across identity subgroups. Under-represented identities receive larger per-sample weights during training so their toxic-rate pattern is normalised toward the corpus mean.

| § | Step |
|---|------|
| 1 | Load data (split_ids.json) |
| 2 | Train BPE tokenizer |
| 3 | Compute importance weights |
| 4 | Model (same GRU as baseline) |
| 5 | Training with weighted BCE loss |
| 6 | Inference & overall performance |
| 7 | Fairness evaluation |
| 8 | Counterfactual gap |

> **Authorship note.** Code generation and prose editing in this notebook were assisted by Claude Opus. Method selection, limitation analysis, and the substance of result interpretations were completed by the human author.

## 0. Setup

In [14]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report, roc_auc_score
from transformers import BertTokenizer

from fairness_jigsaw.metrics import DEFAULT_IDENTITY_COLUMNS, ModelBiasEvaluator

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

TOXICITY_THRESHOLD = 0.5
IDENTITY_THRESHOLD = 0.5
N_TRAIN_SAMPLE     = None   # set to int (e.g. 200_000) to subsample for speed

MAX_LEN    = 128
EMBED_DIM  = 64
HIDDEN_DIM = 128
BATCH_SIZE = 256
EPOCHS     = 3
LR         = 1e-3

Device: mps


## 1. Data

Same pre-defined splits as the baseline (`data/split_ids.json`, seed 1337). Set `N_TRAIN_SAMPLE` to subsample for faster iteration.

In [15]:
with open("../data/split_ids.json") as f:
    split_ids = json.load(f)

USE_COLS = ["id", "target", "comment_text"] + list(DEFAULT_IDENTITY_COLUMNS)
df_all = pd.read_csv("../data/train.csv", usecols=USE_COLS)
df_all["toxic"] = (df_all["target"] >= TOXICITY_THRESHOLD).astype(int)

train_df = df_all[df_all["id"].isin(split_ids["train"])].reset_index(drop=True)
val_df   = df_all[df_all["id"].isin(split_ids["val"])].reset_index(drop=True)
test_df  = df_all[df_all["id"].isin(split_ids["test"])].reset_index(drop=True)

if N_TRAIN_SAMPLE is not None:
    train_df = train_df.sample(n=N_TRAIN_SAMPLE, random_state=SEED).reset_index(drop=True)

for name, df in [("Train", train_df), ("Val  ", val_df), ("Test ", test_df)]:
    anno = df[list(DEFAULT_IDENTITY_COLUMNS)].notna().any(axis=1).sum()
    print(f"{name}: {len(df):>10,}  | toxic: {df['toxic'].mean():.2%}"
          f"  | annotated: {anno:>7,} ({anno/len(df):.1%})")

Train:  1,443,897  | toxic: 8.00%  | annotated: 324,097 (22.4%)
Val  :    180,486  | toxic: 8.00%  | annotated:  40,486 (22.4%)
Test :    180,491  | toxic: 8.00%  | annotated:  40,547 (22.5%)


## 2. Text Preprocessing

Load a **BERT WordPiece** tokenizer (`bert-base-uncased`). Sequences are truncated or padded to `MAX_LEN`.

In [16]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

print(f"BERT vocabulary size : {tokenizer.vocab_size:,} tokens")
print(f"Example tokens       : {tokenizer.tokenize('I hate this person')}")


def encode(text) -> list[int]:
    return tokenizer.encode(
        text if isinstance(text, str) else "",
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )


class CommentDataset(Dataset):
    def __init__(self, df: pd.DataFrame) -> None:
        self.x = torch.tensor([encode(t) for t in df["comment_text"]], dtype=torch.long)
        self.y = torch.tensor(df["toxic"].values, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


test_loader = DataLoader(CommentDataset(test_df), batch_size=BATCH_SIZE)
print(f"Test batches: {len(test_loader)}")

BERT vocabulary size : 30,522 tokens
Example tokens       : ['i', 'hate', 'this', 'person']
Test batches: 706


## 3. Importance Weights

**Method.** For each identity group *c*, compute two label-conditional weights that align `p(toxic | identity = c)` toward the corpus-mean toxic rate:

- **w_toxic(c)** = `tox_mean / tox_rate(c)` — downscales toxic loss for over-toxic groups, upscales for under-toxic ones.  
- **w_nontoxic(c)** = `(1 − tox_mean) / (1 − tox_rate(c))` — upscales non-toxic loss for over-toxic groups.

Each comment is assigned the label-appropriate weight, then the **maximum over all identity groups** the comment belongs to is taken (max-rule aggregation). Comments with no identity annotation keep weight 1.0. Weights are clipped to [0.1, 10.0].

In [17]:
# Binarise identity columns
for col in DEFAULT_IDENTITY_COLUMNS:
    train_df[f"{col}_bin"] = (train_df[col].fillna(0) >= IDENTITY_THRESHOLD).astype(int)


def compute_importance_weights(
    df: pd.DataFrame,
    identity_cols: list,
    label_col: str = "toxic",
    clip: tuple = (0.1, 10.0),
) -> tuple:
    """Toxicity-rate reweighting: aligns p(toxic|identity) toward the corpus mean.

    w_toxic(c)    = tox_mean / tox_rate(c)
    w_nontoxic(c) = (1 - tox_mean) / (1 - tox_rate(c))

    Per-comment weight = max over belonging identities of the label-appropriate weight.
    Max-rule edge case: a comment belonging to both a high-tox and a low-tox group picks
    the higher weight; this is rare and acceptable given max-rule simplicity.
    """
    tox_mean = float(df[label_col].mean())
    labels   = df[label_col].to_numpy()

    records, wtox_list, wntox_list = [], [], []
    for col in identity_cols:
        mask = df[f"{col}_bin"].astype(bool)
        n    = int(mask.sum())
        if n == 0:
            tox_rate, wtox, wntox = float("nan"), 1.0, 1.0
        else:
            tox_rate = float(df.loc[mask, label_col].mean())
            wtox  = float(np.clip(tox_mean / tox_rate           if tox_rate > 0   else 1.0, *clip))
            wntox = float(np.clip((1 - tox_mean) / (1 - tox_rate) if tox_rate < 1.0 else 1.0, *clip))
        records.append({"identity": col, "tox_rate": tox_rate,
                        "tox_target": tox_mean, "w_toxic": wtox, "w_nontoxic": wntox})
        wtox_list.append(wtox)
        wntox_list.append(wntox)

    iw_df = (pd.DataFrame(records)
               .sort_values("tox_rate", ascending=False)
               .reset_index(drop=True))

    bin_mat   = df[[f"{col}_bin" for col in identity_cols]].to_numpy(dtype=np.float32)
    has_id    = bin_mat.max(axis=1) > 0
    wtox_arr  = np.array(wtox_list,  dtype=np.float32)
    wntox_arr = np.array(wntox_list, dtype=np.float32)

    w_arr       = np.where(labels[:, None] == 1, wtox_arr, wntox_arr)  # (N, C)
    per_comment = np.where(has_id, (bin_mat * w_arr).max(axis=1), 1.0).astype(np.float32)
    return iw_df, per_comment


iw_df, train_weights = compute_importance_weights(
    train_df, list(DEFAULT_IDENTITY_COLUMNS), label_col="toxic"
)

print("Identity toxicity-rate weights (sorted by tox_rate desc):")
print(iw_df.to_string(index=False,
      formatters={"tox_rate":   "{:.4%}".format,
                  "tox_target": "{:.4%}".format,
                  "w_toxic":    "{:.4f}".format,
                  "w_nontoxic": "{:.4f}".format}))
print()
print("Per-comment weight distribution:")
print(pd.Series(train_weights).describe(percentiles=[0.5, 0.9, 0.99]).to_string())

Identity toxicity-rate weights (sorted by tox_rate desc):
                           identity tox_rate tox_target w_toxic w_nontoxic
                              black 31.5341%    7.9968%  0.2536     1.3438
                       other_gender 30.0000%    7.9968%  0.2666     1.3143
          homosexual_gay_or_lesbian 28.5131%    7.9968%  0.2805     1.2870
                              white 27.9214%    7.9968%  0.2864     1.2764
           other_sexual_orientation 25.0000%    7.9968%  0.3199     1.2267
                       heterosexual 23.2092%    7.9968%  0.3446     1.1981
                             muslim 22.7573%    7.9968%  0.3514     1.1911
                        transgender 21.5347%    7.9968%  0.3713     1.1725
      psychiatric_or_mental_illness 20.8312%    7.9968%  0.3839     1.1621
                           bisexual 20.5882%    7.9968%  0.3884     1.1586
                   other_disability 20.0000%    7.9968%  0.3998     1.1500
            other_race_or_ethnicity 19.248

## 4. Model

Same single-layer GRU as the baseline — reweighting requires no architectural change.

```
Embedding(|V|, d_e)  ->  GRU(d_e, d_h)  ->  Linear(d_h, 1)  ->  Sigmoid
```

In [18]:
class ToxicityGRU(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru  = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x)
        _, h = self.gru(x)
        return torch.sigmoid(self.head(h[-1])).squeeze(-1)


model    = ToxicityGRU(tokenizer.vocab_size, EMBED_DIM, HIDDEN_DIM).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {n_params:,}")

ToxicityGRU(
  (embedding): Embedding(30522, 64, padding_idx=0)
  (gru): GRU(64, 128, batch_first=True)
  (head): Linear(in_features=128, out_features=1, bias=True)
)

Total parameters: 2,028,033


## 5. Training

Per-sample weights are passed to `F.binary_cross_entropy(weight=w)`, scaling each example's loss contribution before the batch mean is taken.

In [19]:
class WeightedCommentDataset(Dataset):
    def __init__(self, df: pd.DataFrame, weights: np.ndarray) -> None:
        self.x = torch.tensor([encode(t) for t in df["comment_text"]], dtype=torch.long)
        self.y = torch.tensor(df["toxic"].values, dtype=torch.float32)
        self.w = torch.tensor(weights, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx], self.w[idx]


train_loader = DataLoader(
    WeightedCommentDataset(train_df, train_weights),
    batch_size=BATCH_SIZE, shuffle=True,
)
print(f"Train batches: {len(train_loader)}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n_batches = 0.0, 0
    for x, y, w in train_loader:
        x, y, w = x.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        optimizer.zero_grad()
        loss = F.binary_cross_entropy(model(x), y, weight=w)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1
    print(f"Epoch {epoch}/{EPOCHS}  avg_loss={total_loss / n_batches:.4f}")

Train batches: 5641
Epoch 1/3  avg_loss=0.1544
Epoch 2/3  avg_loss=0.1198
Epoch 3/3  avg_loss=0.1126


## 6. Inference & Overall Performance

In [20]:
model.eval()
raw_scores: list[float] = []
with torch.no_grad():
    for x, _ in test_loader:
        raw_scores.extend(model(x.to(DEVICE)).cpu().tolist())

test_df = test_df.copy()
test_df["score"] = raw_scores

overall_auc = roc_auc_score(test_df["toxic"], test_df["score"])
pred_labels = (test_df["score"] >= TOXICITY_THRESHOLD).astype(int)

print(f"Overall AUC          : {overall_auc:.4f}")
print(f"Predicted toxic rate : {pred_labels.mean():.2%}")
print(f"True toxic rate      : {test_df['toxic'].mean():.2%}\n")
print(classification_report(test_df["toxic"], pred_labels, target_names=["non-toxic", "toxic"]))

Overall AUC          : 0.9524
Predicted toxic rate : 4.80%
True toxic rate      : 8.00%

              precision    recall  f1-score   support

   non-toxic       0.96      0.99      0.97    166056
       toxic       0.82      0.49      0.61     14435

    accuracy                           0.95    180491
   macro avg       0.89      0.74      0.79    180491
weighted avg       0.95      0.95      0.94    180491



## 7. Fairness Evaluation

`ModelBiasEvaluator` computes subgroup AUC variants, FPR, and ECE on the test set. Compare to `01_baseline_gru.ipynb` §6 to assess improvement.

| Metric | What it captures |
|--------|------------------|
| **Subgroup / BPSN / BNSP / Pinned AUC** | Discrimination quality per identity |
| **Subgroup FPR (+ gap)** | Over-flagging of non-toxic comments |
| **ECE** | Calibration quality |

Tables sorted worst-first.

In [21]:
evaluator = ModelBiasEvaluator(
    identity_cols=DEFAULT_IDENTITY_COLUMNS,
    toxicity_threshold=TOXICITY_THRESHOLD,
    identity_threshold=IDENTITY_THRESHOLD,
    min_subgroup_size=20,
)

results = evaluator.evaluate(test_df, score_col="score", label_col="toxic")

### AUC Metrics

Low BPSN AUC -> over-predicts toxicity for the group; low BNSP AUC -> under-predicts.

In [22]:
results["auc"].round(4)

,identity,n,subgroup_auc,bpsn_auc,bnsp_auc,pinned_auc
0,other_religion,34,0.7034,0.9143,0.8623,0.7946
1,hindu,55,0.7128,0.9142,0.8720,0.8030
2,buddhist,49,0.7907,0.9030,0.9168,0.8582
3,black,1496,0.8053,0.8740,0.9333,0.8614
4,other_race_or_ethnicity,42,0.8041,0.8999,0.9062,0.8619
5,homosexual_gay_or_lesbian,1141,0.8147,0.8630,0.9456,0.8649
6,transgender,255,0.8258,0.8782,0.9365,0.8733
7,muslim,2151,0.8261,0.8854,0.9354,0.8755
8,white,2551,0.8254,0.8863,0.9361,0.8756
9,heterosexual,116,0.8290,0.9101,0.9181,0.8799


### Subgroup FPR

Positive `fpr_gap` = model over-triggers on non-toxic content from that identity.

In [23]:
results["fpr"].round(4)

,identity,n_negatives,fpr,bg_fpr,fpr_gap
0,transgender,206,0.0291,0.0095,0.0196
1,other_race_or_ethnicity,35,0.0286,0.0095,0.0191
2,atheist,108,0.0278,0.0095,0.0183
3,psychiatric_or_mental_illness,365,0.0274,0.0095,0.0179
4,black,1039,0.0183,0.0094,0.0088
5,homosexual_gay_or_lesbian,819,0.0171,0.0095,0.0076
6,male,3801,0.0142,0.0094,0.0048
7,latino,166,0.0120,0.0095,0.0025
8,white,1839,0.0120,0.0095,0.0025
9,female,4686,0.0109,0.0095,0.0014


### Expected Calibration Error

First row is overall ECE. Subgroup ECE well above overall indicates miscalibration.

In [24]:
results["ece"].round(4)

,identity,n,ece
0,black,1496,0.1607
1,white,2551,0.1429
2,homosexual_gay_or_lesbian,1141,0.1328
3,heterosexual,116,0.1327
4,other_race_or_ethnicity,42,0.1232
5,latino,209,0.1111
6,muslim,2151,0.1053
7,hindu,55,0.1015
8,other_religion,34,0.0952
9,transgender,255,0.0914


## 8. Counterfactual Gap

Neutral substitution (identity -> `"person"`) measures the absolute effect of mentioning an identity after reweighting. Compare to baseline to check whether identity-driven score shifts have reduced.

> **Note.** Only single-word identity terms that appear literally in text are matched. Compound column names (e.g. `homosexual_gay_or_lesbian`) are silently skipped.

In [25]:
def predict_fn(texts: list[str]) -> np.ndarray:
    ids = torch.tensor([encode(t) for t in texts], dtype=torch.long)
    model.eval()
    chunks: list[np.ndarray] = []
    with torch.no_grad():
        for i in range(0, len(ids), BATCH_SIZE):
            chunks.append(model(ids[i : i + BATCH_SIZE].to(DEVICE)).cpu().numpy())
    return np.concatenate(chunks)

cf_results = evaluator.compute_counterfactual_gap(
    test_df,
    text_col   = "comment_text",
    score_col  = "score",
    predict_fn = predict_fn,
    swap_pairs = [("black", "white"), ("christian", "muslim"), ("male", "female")],
)
cf_results.round(4)

,type,term_a,term_b,n_pairs,mean_gap,max_gap
0,neutral,transgender,person,160,0.0421,0.3489
1,neutral,black,person,1917,0.0324,0.5386
2,neutral,muslim,person,1046,0.0284,0.3295
3,neutral,white,person,3875,0.0220,0.4137
4,neutral,christian,person,991,0.0198,0.3185
5,neutral,hindu,person,33,0.0125,0.2275
6,neutral,buddhist,person,28,0.0074,0.0403
7,neutral,jewish,person,339,0.0071,0.0881
8,neutral,male,person,827,0.0054,0.1596
9,neutral,asian,person,212,0.0045,0.1639


## Conclusion

**Metric averages across 18 subgroups (vs. baseline).**

| Metric | Baseline | Reweighting | Δ |
|--------|:--------:|:-----------:|:---:|
| Subgroup AUC | 0.847 | 0.837 | −0.010 |
| BPSN AUC | 0.876 | **0.904** | +0.028 |
| BNSP AUC | 0.946 | 0.926 | −0.020 |
| Pinned AUC | 0.881 | 0.881 | 0.000 |
| FPR | 0.033 | **0.013** | −0.021 |
| FPR gap | 0.022 | **0.003** | −0.019 |
| Subgroup ECE | 0.052 | 0.089 | −0.038 |
| Neutral CF gap | 0.028 | **0.014** | −0.014 |

**Selected identity metrics (Baseline / Reweighting; bold = better).**

| Identity | BPSN AUC | FPR gap | ECE |
|----------|:--------:|:-------:|:---:|
| `black` | 0.806 / **0.874** | +0.059 / **+0.009** | **0.059** / 0.161 |
| `homosexual_gay_or_lesbian` | 0.806 / **0.863** | +0.051 / **+0.008** | **0.038** / 0.133 |
| `transgender` | 0.845 / **0.878** | +0.051 / **+0.020** | **0.062** / 0.091 |
| `muslim` | 0.840 / **0.885** | +0.022 / **−0.002** | **0.034** / 0.105 |
| `white` | 0.823 / **0.886** | +0.034 / **+0.003** | **0.051** / 0.143 |
| `christian` | 0.932 / **0.934** | +0.003 / **−0.003** | **0.015** / 0.026 |

Overall AUC 0.9524 (vs. 0.9540) · Predicted toxic rate 4.80% · Overall ECE 0.012 (vs. 0.006)

---

**Overall performance.** AUC = 0.9524 (−0.002 vs. baseline). Predicted toxic rate 4.80% vs. true 8.00% — globally downscaling toxic-comment losses shifts predicted probabilities downward, hurting recall (49% vs. 53%) while keeping precision high (82%).

**AUC.** Worst pinned AUC: `other_religion` (0.795) and `hindu` (0.803), both small-n. Avg BPSN AUC improves (+0.028), reflecting less over-prediction of non-toxic identity comments. Avg BNSP AUC declines (−0.020), consistent with the model becoming more conservative overall.

**FPR.** Avg FPR drops sharply (0.033 → 0.013) and avg FPR gap from 0.022 to 0.003. Residual over-triggering: `transgender` (gap = +0.020), `other_race_or_ethnicity` (+0.019), `atheist` (+0.018). Groups with negative gaps (`muslim`, `christian`, `jewish`) are under-triggered due to the model's conservatism, not genuine debiasing.

**ECE.** Avg subgroup ECE worsens (0.052 → 0.089). Worst: `black` (0.161), `white` (0.143), `homosexual_gay_or_lesbian` (0.133) — the groups with the highest true toxic rates. Downscaling their toxic-comment losses widens the gap between predicted scores and actual rates.

**Counterfactual gap.** Avg neutral CF gap halves (0.028 → 0.014). Residual associations persist: `transgender` → `person` 0.042; `black` → `person` 0.032. The GRU embedding encodes identity–toxicity co-occurrence that loss reweighting cannot reach.

**Takeaway.** Reweighting corrects the *gradient signal* but leaves the embedding space unchanged. FPR, BPSN AUC, and CF gap improve substantially vs. baseline; subgroup ECE worsens. The root cause — identity-predictive features in the encoder — remains.

**Next step.** `05_mitigation_gru_inprocessing.ipynb` applies adversarial debiasing to suppress identity-predictive signals directly from the GRU hidden representation.